# Flower Image Classification with Transfer Learning (Keras)

The goal of this project is a model that classifies flowers into categories based on photos.

## Approach

This is implemented using **transfer learning**: a network pre-trained on ImageNet (**MobileNetV2**) serves as the foundation. Its weights are frozen, and only a new classification head is trained on the flower dataset.

## Data

The data source is the TensorFlow dataset **`flower_photos`** with five classes. The data is split reproducibly into training and validation subsets (fixed seed) to ensure a non-overlapping and therefore meaningful evaluation.

## Evaluation

Model performance is assessed using:

- Validation accuracy
- Training and loss curves
- Confusion matrix

## Note

> The concept was developed with the support of Claude; all code was evaluated, reviewed, and adapted by me.

In [40]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np

### Data Initialization & Data Format

In [41]:
img_size = (160,160)
bat_size = 32
rounds = 8
SEED = 42

data_url = ("https://storage.googleapis.com/download.tensorflow.org/"
            "example_images/flower_photos.tgz")
data_dir = keras.utils.get_file(
    origin=data_url,#Download Files from
    fname="flower_photos",#Names File
    untar=True,#open File
)
print("Data:", data_dir)


Data: C:\Users\Admin\.keras\datasets\flower_photos


### Validation and Training Split

In [42]:
train_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=img_size,
    batch_size=rounds,
    label_mode="int",
)
val_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=img_size,
    batch_size=rounds,
    label_mode="int",
)
class_names = train_ds.class_names
num_classes = len(class_names)
print("Klassen:", class_names)
 
# Performance: Prefetching for less latenzy
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


Found 3670 files belonging to 1 classes.
Using 2936 files for training.
Found 3670 files belonging to 1 classes.
Using 734 files for validation.
Klassen: ['flower_photos']


### Create Modell for Transfer Learning 

In [ ]:
#prevent Overfitting
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="augmentation")

#load Modell MobileNEtV2 in Format
base_model = keras.applications.MobileNetV2(input_shape=img_size+(3,),
include_top = False,# include_top=False -> drop its old decision layer (we don't need it).
weights = "imaganet",# weights="imagenet" -> bring along its trained "vision" (learned features).
)

base_model.trainable = False #trainable = False -> freeze that knowledge so training leaves it unchanged

input = keras.Input(shape=img_size+(3,))
x = data_augmentation(input)
x = keras.applications.mobilenet_v2.preprocess_input_(x) # f: [0,255] -> [-1,1]
x = base_model(x,training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes)(x)